In [10]:
import os
import csv
import glob
import logging
import tempfile
import subprocess
from pathlib import Path
from typing import Tuple

import pdfplumber
from pypdf import PdfReader

## Logging Setup

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger(__name__)

### Config path

In [12]:
DATASET_DIR      = Path("..\Dataset")       # root containing occupation subfolders
OUTPUT_DIR       = Path("..\Extracted_data")     # mirrored output tree (.txt files)
LOG_FILE         = Path("..\extraction_log.csv")
MIN_CHARS_NATIVE = 50                    # char threshold to classify as native PDF
OCR_DPI          = 200                   # DPI for rasterising scanned pages

### Detection

In [13]:
def is_native_pdf(pdf_path: Path, sample_pages: int = 2) -> bool:
    """
    Returns True if the PDF has a readable text layer (native/text-based).
    Samples the first `sample_pages` pages; if extracted text exceeds
    MIN_CHARS_NATIVE characters, it is treated as native.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "".join(
                (page.extract_text() or "") for page in pdf.pages[:sample_pages]
            )
        return len(text.strip()) >= MIN_CHARS_NATIVE
    except Exception:
        return False  # if we can't read it at all, fall back to OCR


### PDF Extraction

In [14]:
def extract_native(pdf_path: Path) -> Tuple[str, int]:
    """
    Direct text-layer extraction using pdfplumber.
    Returns (full_text, page_count).
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            t = page.extract_text() or ""
            if t.strip():
                pages_text.append(f"--- Page {i + 1} ---\n{t}")
    return "\n\n".join(pages_text), page_count

### Master extraction

In [15]:
def extract_pdf(pdf_path: Path) -> dict:
    """
    Detects PDF type and routes to the appropriate extractor.
    Returns a result dict including the extracted text and metadata.
    """
    result = {
        "occupation": pdf_path.parent.name,
        "filename":   pdf_path.name,
        "file":       str(pdf_path),
        "method":     None,
        "page_count": 0,
        "char_count": 0,
        "status":     "ok",
        "error":      "",
        "text":       "",
    }

    try:
        native = is_native_pdf(pdf_path)
        if native:
            text, pages = extract_native(pdf_path)
            result["method"] = "native_pdfplumber"
        else:
            log.info(f"-> Scanned PDF detected, applying OCR: {pdf_path.name}")
            text, pages = extract_scanned(pdf_path)
            result["method"] = "ocr_pytesseract"

        result["text"]       = text
        result["page_count"] = pages
        result["char_count"] = len(text)

        if not text.strip():
            result["status"] = "empty"
            result["error"]  = "No text could be extracted"

    except Exception as exc:
        result["status"] = "error"
        result["error"]  = str(exc)
        log.error(f"Error processing {pdf_path.name}: {exc}")

    return result